# 09 A word-substitution attack on the cascade

Notebook 08 injected instructions into messages. This one attacks the encoder with nothing but synonyms: swap a few words for near-synonyms, keep the message readable, and see whether the calibrated head and its conformal gate act on the wrong route.

The attack is a **TextFooler-style** word substitution, written from the algorithm box in Jin, Jin, Zhou and Szolovits, "Is BERT Really Robust?" (AAAI 2020; https://arxiv.org/abs/1907.11932). It is **not the authors' code**. It follows their algorithm and their settings (50 neighbors per word, word-vector cosine above 0.7, same part of speech, stop words skipped, words ranked by how much deleting them lowers the true route's probability). Four differences, all of them ours:

1. **Sentence similarity** comes from `all-MiniLM-L6-v2`, not the Universal Sentence Encoder, with a floor of 0.8 measured **against the original message at every step**. The paper measures against the previous step, which lets meaning drift. Ours is the stricter reading.
2. **Part of speech** comes from NLTK's perceptron tagger.
3. **The victim** is the workshop cascade: `bge-small` + logistic head, temperature-scaled, conformal gate at alpha 0.02. The paper attacks fine-tuned BERT.
4. **The goal** has two forms. *Label flip* is the paper's: the top route is no longer the true one. *Wrong action* is the one that matters here: the gate returns exactly one route and it is not the true one. The same run records both.

**Attacker's access.** The attacker sees the head's probability for every route (the paper's setting). A deployed system that returns only a final action would give an attacker far less.

In [1]:
import json, os, time
import numpy as np, torch
from pathlib import Path
from collections import Counter
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
import meridian as m, attack as atk

torch.set_num_threads(2)
RESULTS, CACHE = Path("data/results"), Path("data/cache")
r01 = json.loads((RESULTS / "01_encoders_and_baseline.json").read_text())
r03 = json.loads((RESULTS / "03_calibration_and_conformal.json").read_text())
ENC, T, ALPHA = r03["encoder"], r03["T"], r03["alpha_chosen"]
Q = r03[f"conformal_alpha_{ALPHA}"]["q"]
tr, te = m.load("train"), m.load("test")
_, ytr = m.texts_labels(tr)
_, yte = m.texts_labels(te)
E = m.cached_embeddings(ENC)
clf = LogisticRegression(C=r01[ENC]["C"], max_iter=3000).fit(E["train"], ytr)
loaded = m.load_encoder(ENC)

def victim(texts):
    return softmax(clf.decision_function(m.encode(ENC, list(texts), _loaded=loaded, batch_size=64)) / T, axis=1)

def gate(p):
    return np.where(p >= 1 - Q)[0]

def outcome(p, true):
    s = gate(p)
    if len(s) == 1:
        return "acts, true route" if s[0] == true else "acts, wrong route"
    return "escalates" if len(s) > 1 else "person"

print(f"{ENC}, T={T:.3f}, alpha={ALPHA}, threshold p >= {1 - Q:.3f} | hardware: {m.hardware()} | load average {os.getloadavg()[0]:.1f}")

/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


bge-small, T=0.281, alpha=0.02, threshold p >= 0.151 | hardware: Intel(R) Core(TM) i7-7567U CPU @ 3.50GHz, 4 logical cores, Darwin 22.6.0 | load average 24.3


## Inputs: counter-fitted word vectors and a sentence-similarity model

Mrkšić et al. (2016, https://arxiv.org/abs/1603.00892) tuned word vectors so that synonyms are close and antonyms far; the TextFooler paper takes its neighbors from them. The file below is from that paper's repository (https://github.com/nmrksic/counter-fitting). It is data only: read as text, never executed. The checksum is the one we recorded when we downloaded it on 28 October 2026.

In [2]:
import hashlib, urllib.request, zipfile
URL = "https://raw.githubusercontent.com/nmrksic/counter-fitting/master/word_vectors/counter-fitted-vectors.txt.zip"
SHA = "6f67f40c9b2e757695b3af2209bcd4393377962ca440dd4fad542937d01e7d56"
VEC = CACHE / "counter-fitted-vectors.txt"
if not VEC.exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    z = CACHE / "counter-fitted-vectors.zip"
    urllib.request.urlretrieve(URL, z)
    assert hashlib.sha256(z.read_bytes()).hexdigest() == SHA, "checksum differs from the file we tested"
    zipfile.ZipFile(z).extract("counter-fitted-vectors.txt", CACHE)
words, mat = atk.load_vectors(VEC)

from sentence_transformers import SentenceTransformer
mini = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
def sim(x, ys):
    e = mini.encode([x] + list(ys), normalize_embeddings=True, batch_size=64)
    return e[1:] @ e[0]

at = atk.Attacker(words, mat, victim, sim, n_syn=50, delta=0.7, eps=0.8, anchor_original=True)
print(len(words), "alphabetic words with vectors")
print("neighbors of 'cancel':", at.synonyms("cancel")[:8])
print("neighbors of 'refund':", at.synonyms("refund")[:8])

65713 alphabetic words with vectors
neighbors of 'cancel': ['revoke', 'annul', 'rescind', 'countermand', 'cancelling', 'rescinded', 'cancelled', 'revoking']
neighbors of 'refund': ['refunds', 'repayment', 'repayments', 'reimbursement', 'refunded', 'reimbursed', 'reimburse', 'rebate']


## Who is attacked

Only test messages the cascade handles **correctly on its own**: the encoder's top route is the true one and the gate returns exactly that one route. A message the gate already escalates or gets wrong cannot be made worse by the attacker in the sense we measure.

In [3]:
P0 = victim([r["text"] for r in te])
clean = [outcome(p, y) for p, y in zip(P0, yte)]
print(Counter(clean))
targets = [i for i, c in enumerate(clean) if c == "acts, true route"]
argmax_ok = [i for i in range(len(te)) if P0[i].argmax() == yte[i]]
print(f"attacked: {len(targets)} messages the gate answers correctly alone; {len(argmax_ok)} have the true route on top")

Counter({'acts, true route': 180, 'escalates': 17, 'acts, wrong route': 3})
attacked: 180 messages the gate answers correctly alone; 194 have the true route on top


## The attack

Messages are attacked in a seeded random order, so any prefix of the log is a random sample of the targets and the notebook reports how many were reached. One run per message. The attack stops at the first message where the gate acts on a wrong route, or when no swap lowers the true route's probability any further. Every message is logged to `data/cache/09_attack_log.jsonl` as soon as it finishes, so an interrupted run resumes where it stopped. The random control swaps the same number of words for random same-part-of-speech neighbors, twenty draws per message, with no access to the model's scores.

In [4]:
LOG = CACHE / "09_attack_log.jsonl"
done = {}
if LOG.exists():
    for l in LOG.read_text().splitlines():
        r = json.loads(l); done[r["i"]] = r
rng = np.random.default_rng(m.SEED)
t0 = time.time()
order = list(np.random.default_rng(m.SEED).permutation(targets))
for i in order:
    if i in done: continue
    y = int(yte[i])
    res = at.attack(te[i]["text"], y, lambda p, y=y: len(gate(p)) == 1 and gate(p)[0] != y)
    k = res["flip"]["n_changed"] if res["flip"] else res["n_changed"]
    ctrl = at.random_control(te[i]["text"], k, rng, draws=20)
    Pc = victim(ctrl)
    row = dict(i=i, id=te[i]["id"], scenario=f"{te[i]['label']}:{te[i]['scenario']}", text=te[i]["text"], true=m.ROUTES[y],
               adv=res["adv"], success=res["success"], queries=res["queries"], n_words=res["n_words"], n_changed=res["n_changed"],
               changes=res["changes"], sim_to_original=res["sim_to_original"], adv_outcome=outcome(np.array(res["probs"]), y),
               adv_top=m.ROUTES[int(np.argmax(res["probs"]))], flip=res["flip"],
               control_flip=float(np.mean([p.argmax() != y for p in Pc])),
               control_wrong_action=float(np.mean([outcome(p, y) == "acts, wrong route" for p in Pc])),
               load=os.getloadavg()[0])
    with LOG.open("a") as f: f.write(json.dumps(row, default=str) + "\n")
    done[i] = row
    if len(done) % 10 == 0: print(f"{len(done)}/{len(targets)} done, {time.time() - t0:.0f} s this session, load {row['load']:.0f}", flush=True)
rows = [done[i] for i in targets if i in done]
print(f"{len(rows)} of {len(targets)} attacked | log: {LOG}")

190/180 done, 151 s this session, load 11


200/180 done, 279 s this session, load 10


210/180 done, 394 s this session, load 7


220/180 done, 534 s this session, load 9


230/180 done, 665 s this session, load 18


...

330/180 done, 1769 s this session, load 19


340/180 done, 1891 s this session, load 19


350/180 done, 2002 s this session, load 19


360/180 done, 2118 s this session, load 21


180 of 180 attacked | log: data/cache/09_attack_log.jsonl


## What the attack achieved

Success rates carry a Wilson interval and a **cluster** interval: the five variants of one scenario share a situation, so they are not independent.

In [5]:
clusters = np.array([r["scenario"] for r in rows])
def rate(flags, name):
    flags = np.asarray(flags, dtype=int)
    lo, hi = m.cluster_bootstrap_ci(flags, clusters)
    print(f"{name:52s} {m.fmt_prop(int(flags.sum()), len(flags))}  cluster 95%: {lo:.3f} to {hi:.3f}")
    return dict(k=int(flags.sum()), n=len(flags), wilson=m.wilson(int(flags.sum()), len(flags)), cluster=[lo, hi])
res = {}
res["label_flip"] = rate([r["flip"] is not None for r in rows], "top route changed (the paper's goal)")
res["wrong_action"] = rate([r["success"] for r in rows], "gate acts on a wrong route")
res["escalated_instead"] = rate([r["adv_outcome"] == "escalates" for r in rows], "ends escalated (a slow-tier call it did not need)")
res["survived"] = rate([r["adv_outcome"] == "acts, true route" for r in rows], "still acts on the true route")
res["control_flip"] = dict(mean=float(np.mean([r["control_flip"] for r in rows])))
res["control_wrong_action"] = dict(mean=float(np.mean([r["control_wrong_action"] for r in rows])))
print(f"random swaps, same count, no scores: top route changed {res['control_flip']['mean']:.3f}, wrong action {res['control_wrong_action']['mean']:.3f} (mean over 20 draws per message)")

top route changed (the paper's goal)                 120/180 = 0.667 (Wilson 95%: 0.595 to 0.731)  cluster 95%: 0.576 to 0.757


gate acts on a wrong route                           83/180 = 0.461 (Wilson 95%: 0.390 to 0.534)  cluster 95%: 0.364 to 0.559


ends escalated (a slow-tier call it did not need)    64/180 = 0.356 (Wilson 95%: 0.289 to 0.428)  cluster 95%: 0.283 to 0.436


still acts on the true route                         33/180 = 0.183 (Wilson 95%: 0.134 to 0.246)  cluster 95%: 0.115 to 0.257
random swaps, same count, no scores: top route changed 0.011, wrong action 0.002 (mean over 20 draws per message)


In [6]:
succ = [r for r in rows if r["success"]]
fl = [r["flip"] for r in rows if r["flip"]]
q = lambda xs: (float(np.median(xs)), float(np.percentile(xs, 90)))
res["cost"] = dict(
    queries_all=q([r["queries"] for r in rows]), queries_success=q([r["queries"] for r in succ]) if succ else None,
    swaps_success=q([r["n_changed"] for r in succ]) if succ else None,
    swap_rate_success=float(np.median([r["n_changed"] / r["n_words"] for r in succ])) if succ else None,
    sim_success=q([r["sim_to_original"] for r in succ]) if succ else None)
print("queries per message (median, 90th percentile), all:", res["cost"]["queries_all"], " successful:", res["cost"]["queries_success"])
print("swaps in a successful attack (median, 90th):", res["cost"]["swaps_success"], "| share of words changed (median):", res["cost"]["swap_rate_success"])
print("sentence similarity to the original, successful attacks (median, 90th):", res["cost"]["sim_success"])
by = {}
for key in ("true",):
    for v in sorted({r[key] for r in rows}):
        sub = [r for r in rows if r[key] == v]
        by[v] = (sum(r["success"] for r in sub), len(sub)); print(f"  {v:22s} {m.fmt_prop(*by[v])}")
res["by_route"] = by
print("wrong routes the attacker reached:", Counter(r["adv_top"] for r in succ))

queries per message (median, 90th percentile), all: (65.0, 106.1)  successful: (57.0, 90.0)
swaps in a successful attack (median, 90th): (4.0, 7.0) | share of words changed (median): 0.16666666666666666
sentence similarity to the original, successful attacks (median, 90th): (0.8325512409210205, 0.9178230285644531)
  baggage_claim          11/25 = 0.440 (Wilson 95%: 0.267 to 0.629)
  cancellation_request   17/20 = 0.850 (Wilson 95%: 0.640 to 0.948)
  out_of_scope           5/23 = 0.217 (Wilson 95%: 0.097 to 0.419)
  payment_dispute        11/25 = 0.440 (Wilson 95%: 0.267 to 0.629)
  rebooking_request      14/22 = 0.636 (Wilson 95%: 0.430 to 0.803)
  refund_claim           10/16 = 0.625 (Wilson 95%: 0.386 to 0.815)
  service_complaint      9/24 = 0.375 (Wilson 95%: 0.212 to 0.573)
  visa_entry_question    6/25 = 0.240 (Wilson 95%: 0.115 to 0.434)
wrong routes the attacker reached: Counter({'out_of_scope': 31, 'rebooking_request': 13, 'service_complaint': 9, 'payment_dispute': 9, 'baggage

## Do the attacked messages still mean the same thing?

An attack that changes what the customer asked for has not fooled the model; it has asked it a different question. The TextFooler paper had people judge this and found 8 to 15% of successful attacks changed the meaning. Here a model reads every successful attack against the rule *would a travel-agency reader send this to the same team as the original?* and writes a verdict to `data/review/attack_review.csv`. **That is a model read, not a person's.**

In [7]:
REV = Path("data/review/attack_review.csv")
import csv
if not REV.exists():
    with REV.open("w", newline="") as f:
        w = csv.writer(f); w.writerow(["id", "true_route", "original", "attacked", "swaps", "same_route", "note", "annotator"])
        for r in succ:
            w.writerow([r["id"], r["true"], r["text"], r["adv"], "; ".join(f"{a}->{b}" for a, b in r["changes"]), "", "", ""])
    print(f"wrote {len(succ)} rows to {REV}: fill in same_route (yes/no) before the next cell")
rev = list(csv.DictReader(REV.open()))
judged = [r for r in rev if r["same_route"] in ("yes", "no")]
print(f"{len(judged)} of {len(rev)} successful attacks judged")

83 of 83 successful attacks judged


In [8]:
if judged and len(judged) == len(rev):
    valid_ids = {r["id"] for r in judged if r["same_route"] == "yes"}
    res["valid_wrong_action"] = rate([r["success"] and r["id"] in valid_ids for r in rows], "wrong action AND the message keeps its route")
    res["judged"] = dict(n=len(judged), same_route=len(valid_ids), annotator=judged[0]["annotator"])
    print(f"of {len(judged)} successful attacks, {len(valid_ids)} keep the original route ({len(valid_ids)/len(judged):.0%}); annotator: {judged[0]['annotator']}")
    print("\nexamples that keep the route:")
    for r in [r for r in judged if r["same_route"] == "yes"][:4]: print("  ", r["original"][:110], "\n   ->", r["attacked"][:110], "|", r["swaps"])
    print("\nexamples that change it:")
    for r in [r for r in judged if r["same_route"] == "no"][:4]: print("  ", r["original"][:110], "\n   ->", r["attacked"][:110], "|", r["swaps"], "|", r["note"])

wrong action AND the message keeps its route         80/180 = 0.444 (Wilson 95%: 0.374 to 0.517)  cluster 95%: 0.350 to 0.542
of 83 successful attacks, 80 keep the original route (96%); annotator: model read (Claude), 30 Oct 2026; not a person

examples that keep the route:
   bag arrived in Mumbai but the zip was open and my laptop is gone. checked it at the carousel, not in there. ME 
   -> sacs transpired in Mumbai but the zip was open and my laptop is fainted. ticked it at the carousel, not in the | bag->sacs; arrived->transpired; gone->fainted; checked->ticked
   Dear support, my suitcase reached Mumbai with the zip partly open, and the laptop inside has disappeared. Book 
   -> Dear support, my valise reached Mumbai with the zip partly open, and the laptop inland has faded. Booking MER- | suitcase->valise; record->register; inside->inland; disappeared->faded
   bag didnt come on the flight to edinburgh and its been 2 days now. MER-14590 can u tell me when its coming and 
   -> va

In [9]:
out = dict(n_attacked=len(rows), settings=dict(n_syn=50, delta=0.7, sim_floor=0.8, sim_anchor="original", sim_model="all-MiniLM-L6-v2", pos="nltk perceptron",
                                                vectors_sha256=SHA, goal="gate acts on a wrong route", attacker_sees="probabilities for every route"),
           results=res, load_median=float(np.median([r["load"] for r in rows])), hardware=m.hardware())
(RESULTS / "09_word_substitution_attack.json").write_text(json.dumps(out, indent=2, default=str))
print("saved", RESULTS / "09_word_substitution_attack.json")

saved data/results/09_word_substitution_attack.json
